# Deploy — cardiac-DTI whole-heart segmentation (point a folder, get labels)

Thin wrapper over `deployable.py`. Automatic pipeline:
**DWI → YOLO crop → resize 256 → nnUNet LV → YOLO insertion points → un-crop to original spacing.**

## 1. Your folders
One input folder, per case named `<case>_000X.nii.gz` (co-registered, same shape per case):

```
input/
  Patient01_0000.nii.gz   # DWI  (average diffusion-weighted image)   REQUIRED
  Patient01_0001.nii.gz   # MD   (mean diffusivity)                   baseline 2-contrast
  Patient02_0000.nii.gz
  Patient02_0001.nii.gz
```

## 2. Rules
- Fixed suffixes: `_0000` = DWI, `_0001` = MD (add `_0002…` only if your nnUNet LV model was
  trained with more channels — same order as training).
- `_0000` (DWI) is always required (crop + insertion points run on it).
- Set `CHANNEL_NORM` to match training: `['minmax','md4']` = DWI min-max, MD /4 (the baseline).

## 3. Weights
Set the three paths below (local). To host them, put URLs in `deployable.py`'s `*_URL`
fields and run `download_weights()` once.

## 4. Output
`output/<case>.nii.gz` in the **original spacing**: 1 = LV, 2 = anterior IP, 3 = inferior IP.

In [ ]:
import importlib, deployable
importlib.reload(deployable)
from deployable import CardiacPipeline, download_weights

# ---- edit these ----
INPUT_DIR  = 'input'                 # folder with <case>_0000.nii.gz (+ _0001 ...)
OUTPUT_DIR = 'output'                # segmentations land here

CROP_YOLO    = '/path/to/crop_best.pt'          # YOLO crop model
IP_YOLO      = '/path/to/ip_best.pt'            # YOLO insertion-point model (single-contrast avg)
LV_MODEL_DIR = '/path/to/nnUNet_LV_model_dir'   # nnUNet results folder (plans.json, dataset.json, fold_0 ...)

CHANNEL_NORM = ['minmax', 'md4']     # DWI + MD baseline; match your model's channels/order
# download_weights()                 # <- uncomment once *_URL are set in deployable.py

## Run

In [ ]:
pipe = CardiacPipeline(crop_yolo=CROP_YOLO, ip_yolo=IP_YOLO,
                       lv_model_dir=LV_MODEL_DIR, channel_norm=CHANNEL_NORM)
pipe.predict_folder(INPUT_DIR, OUTPUT_DIR)

## Peek at a result (overlay the DWI + segmentation)

In [ ]:
import glob, os, numpy as np, nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

CMAP = ListedColormap([(0,0,0,0),(1,1,0,0.5),(1,0,0,0.7),(0,0,1,0.7)])  # LV yellow, IP1 red, IP2 blue
segs = sorted(glob.glob(f'{OUTPUT_DIR}/*.nii.gz'))[:6]
if segs:
    fig, ax = plt.subplots(1, len(segs), figsize=(3*len(segs), 3))
    if len(segs)==1: ax=[ax]
    for a, sp in zip(ax, segs):
        case = os.path.basename(sp)[:-7]
        img = nib.load(f'{INPUT_DIR}/{case}_0000.nii.gz').get_fdata().squeeze()
        seg = nib.load(sp).get_fdata().squeeze()
        a.imshow(img, cmap='gray')
        a.imshow(np.ma.masked_where(seg==0, seg), cmap=CMAP, vmin=0, vmax=3, interpolation='none')
        a.set_title(case[:20], fontsize=8); a.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('no outputs yet — run the cell above first')